In [1]:
from pathlib import Path
import pynq
import time

In [2]:
BASE_SIZE = (8 * 2) + (3 * 1)
FIXED_SIZE = (64 * 7) + BASE_SIZE


In [3]:
def load_csv_ints(file_path: Path) -> list[int]:
    values: list[int] = []
    with file_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            values.extend(int(token.strip()) for token in line.split(",") if token.strip())
    return values


def check(labels, recv_buff, n=1):
    correct = True
    n_labels = len(labels)
    for i in range(n):
        for idx, (actual, predicted) in enumerate(zip(labels, recv_buff[i * n_labels:])):
            if (actual != predicted):
                print(f"Mismatch at idx {i * n_labels + idx}: actual={actual}, predicted={predicted}")
                correct = False
    return correct


In [4]:
data_dir = Path("./data")
ordered_files = ["w_hid.csv", "w_out.csv"]

w_hid = load_csv_ints(data_dir / "w_hid.csv")
w_out = load_csv_ints(data_dir / "w_out.csv")
X = load_csv_ints(data_dir / "X.csv")
labels = load_csv_ints(data_dir / "labels.csv")

In [5]:
ov = pynq.Overlay("project_hw.xsa")

In [6]:
names = ["fixed_hls", "hdl"]
dmas = [ov.dma_hls_ip, ov.dma_hdl_ip]
send_chans = [dma.sendchannel for dma in dmas]
recv_chans = [dma.recvchannel for dma in dmas]

In [13]:
send_buff = pynq.allocate(shape=(2048,))
recv_buff = pynq.allocate(shape=(2048,))

In [14]:
print(f"Send buffer address: {hex(send_buff.physical_address)}")
print(f"Recv buffer address: {hex(recv_buff.physical_address)}")

Send buffer address: 0x375b8000
Recv buffer address: 0x375ba000


In [22]:
pay_size = FIXED_SIZE

print("=== 64 x 7 (DEFAULT) X DMA Transaction Comparison ===")
for (name, send, recv) in zip(names, send_chans, recv_chans):
    print(f"=== {name} ===")
    # Initialize buffer
    if "hls" in name:
        temp = [*X, *w_hid, *w_out]
    else:
        temp = [*w_hid, *w_out, *X]

    for i, val in enumerate(temp):
        send_buff[i] = val

    _pay_size = FIXED_SIZE if "fixed" in name else pay_size

    start = time.time()
    recv.transfer(recv_buff)
    send.transfer(send_buff[:_pay_size])
    send.wait()
    recv.wait()
    end = time.time()
    print("--- Checking Results ---")
    correct = check(labels, recv_buff)
    print("--- Summary ---")
    print(f"Transaction duation: {end - start}")
    print(f"Correctness check: {'PASS' if correct else  'FAIL'}")
    print("=== END ===\n")


=== 64 x 7 (DEFAULT) X DMA Transaction Comparison ===
=== fixed_hls ===
--- Checking Results ---
--- Summary ---
Transaction duation: 0.0018575191497802734
Correctness check: PASS
=== END ===

=== hdl ===
--- Checking Results ---
--- Summary ---
Transaction duation: 0.0005757808685302734
Correctness check: PASS
=== END ===



In [ ]:
del send_buff, recv_buff